<a href="https://colab.research.google.com/github/Doran1690/llm-engineering-projects/blob/main/gym_churn_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.7 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from huggingface_hub import login

In [4]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [6]:
MODEL = "meta-llama/Llama-3.2-3B-Instruct"

In [7]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, device_map="auto", quantization_config=quant_config)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [9]:
messages = [
    {"role":"system","content":"You are a helpful assistant who generates realistic synthetic data. The data should look real and varied. Reply only with the data itself, without any explanations."},
    {"role":"user","content":"  Give me 5 customer reviews about a restaurant named Sicily Pizza."}
]

In [10]:
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=500)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(response)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


1. "Hands down the best pizza I've had in years! The Sicilian-style crust was perfect, and the toppings were fresh and flavorful. 10/10 would recommend!" - Emily W.

2. "I was a bit skeptical about trying a new pizza place, but Sicily Pizza did not disappoint. The atmosphere was cozy and intimate, and the service was top-notch. Highly recommend!" - David K.

3. "I'm a huge fan of garlic knots, and Sicily Pizza did not disappoint. They were crispy on the outside and soft on the inside, just like I like them. 5 stars!" - Rachel T.

4. "I ordered the veggie pizza and was a bit disappointed that it didn't have any meat options. However, the veggies were all fresh and the sauce was delicious. Would order again." - James B.

5. "Sicily Pizza has the best cannoli in town! The crust was crispy and the filling was creamy and sweet. A must-try dessert!" - Maria G.


In [11]:
def generate_data(request):
    messages = [
        {"role": "system", "content": "You are a helpful assistant who generates realistic synthetic data. The data should look real and varied. Reply only with the data itself, without any explanations."},
        {"role": "user", "content": request}
    ]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=500)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

In [12]:
print(generate_data("Give me 5 fake customers with name, age and city."))

1. Emily Wilson - 32 - New York
2. David Kim - 41 - Los Angeles
3. Sophia Patel - 28 - Chicago
4. Michael Brown - 55 - Houston
5. Jessica Martin - 19 - Miami


In [13]:
import gradio as gr

gr.Interface(fn=generate_data, inputs="text", outputs="text").launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://838ad68d0d9d9d19b9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [14]:
csv_text = """
member_id,join_date,age,branch,plan,monthly_fee,visits_per_month,personal_trainer,churned
1,2024-06-15,32,Tel Aviv,monthly,250,15,yes,no
2,2024-03-22,41,Ramat Gan,annual,180,5,no,yes
3,2025-01-01,28,Holon,monthly,250,12,no,yes
4,2024-09-10,25,Ramat Gan,student,150,6,no,no
5,2025-05-20,41,Ramat Gan,annual,180,10,yes,yes
6,2024-11-18,29,Tel Aviv,monthly,250,18,yes,no
7,2025-02-25,35,Holon,student,150,14,no,no
8,2024-08-05,24,Ramat Gan,monthly,250,11,yes,yes
9,2024-04-15,28,Tel Aviv,annual,180,7,yes,no
10,2024-10-22,43,Ramat Gan,monthly,250,19,yes,yes
"""
print(csv_text)


member_id,join_date,age,branch,plan,monthly_fee,visits_per_month,personal_trainer,churned
1,2024-06-15,32,Tel Aviv,monthly,250,15,yes,no
2,2024-03-22,41,Ramat Gan,annual,180,5,no,yes
3,2025-01-01,28,Holon,monthly,250,12,no,yes
4,2024-09-10,25,Ramat Gan,student,150,6,no,no
5,2025-05-20,41,Ramat Gan,annual,180,10,yes,yes
6,2024-11-18,29,Tel Aviv,monthly,250,18,yes,no
7,2025-02-25,35,Holon,student,150,14,no,no
8,2024-08-05,24,Ramat Gan,monthly,250,11,yes,yes
9,2024-04-15,28,Tel Aviv,annual,180,7,yes,no
10,2024-10-22,43,Ramat Gan,monthly,250,19,yes,yes



In [26]:
import pandas as pd
from io import StringIO

df = pd.read_csv(StringIO(csv_text))
df

,member_id,join_date,age,branch,plan,monthly_fee,visits_per_month,personal_trainer,churned
0,1,2024-06-15,32,Tel Aviv,monthly,250,15,yes,no
1,2,2024-03-22,41,Ramat Gan,annual,180,5,no,yes
2,3,2025-01-01,28,Holon,monthly,250,12,no,yes
3,4,2024-09-10,25,Ramat Gan,student,150,6,no,no
4,5,2025-05-20,41,Ramat Gan,annual,180,10,yes,yes
5,6,2024-11-18,29,Tel Aviv,monthly,250,18,yes,no
6,7,2025-02-25,35,Holon,student,150,14,no,no
7,8,2024-08-05,24,Ramat Gan,monthly,250,11,yes,yes
8,9,2024-04-15,28,Tel Aviv,annual,180,7,yes,no
9,10,2024-10-22,43,Ramat Gan,monthly,250,19,yes,yes


In [27]:
df["join_date"] = pd.to_datetime(df["join_date"])

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   member_id         10 non-null     int64         
 1   join_date         10 non-null     datetime64[ns]
 2   age               10 non-null     int64         
 3   branch            10 non-null     object        
 4   plan              10 non-null     object        
 5   monthly_fee       10 non-null     int64         
 6   visits_per_month  10 non-null     int64         
 7   personal_trainer  10 non-null     object        
 8   churned           10 non-null     object        
dtypes: datetime64[ns](1), int64(4), object(4)
memory usage: 852.0+ bytes


In [22]:
df["branch"].value_counts()

,count
branch,
Ramat Gan,5
Tel Aviv,3
Holon,2


In [23]:
df["plan"].value_counts()

,count
plan,
monthly,5
annual,3
student,2


In [28]:
df["churned"] = df["churned"].map({"yes": 1, "no": 0})
df

,member_id,join_date,age,branch,plan,monthly_fee,visits_per_month,personal_trainer,churned
0,1,2024-06-15,32,Tel Aviv,monthly,250,15,yes,0
1,2,2024-03-22,41,Ramat Gan,annual,180,5,no,1
2,3,2025-01-01,28,Holon,monthly,250,12,no,1
3,4,2024-09-10,25,Ramat Gan,student,150,6,no,0
4,5,2025-05-20,41,Ramat Gan,annual,180,10,yes,1
5,6,2024-11-18,29,Tel Aviv,monthly,250,18,yes,0
6,7,2025-02-25,35,Holon,student,150,14,no,0
7,8,2024-08-05,24,Ramat Gan,monthly,250,11,yes,1
8,9,2024-04-15,28,Tel Aviv,annual,180,7,yes,0
9,10,2024-10-22,43,Ramat Gan,monthly,250,19,yes,1


In [29]:
df["churned"].mean()

np.float64(0.5)

In [30]:
df.groupby("branch")["churned"].mean()

,churned
branch,
Holon,0.5
Ramat Gan,0.8
Tel Aviv,0.0


In [31]:
request = """
Generate 10 rows of data about gym members.
Output only CSV with a header row. No explanations, no extra text.

Columns:
member_id: numbers starting from 1
join_date: a date between 2024-01-01 and 2025-12-31
age: between 18 and 65
branch: one of Tel Aviv, Ramat Gan, Holon
plan: one of monthly, annual, student
monthly_fee: monthly = 250, annual = 180, student = 150
visits_per_month: between 0 and 20
personal_trainer: yes or no
"""

answer = generate_data(request)
print(answer)

"member_id","join_date","age","branch","plan","monthly_fee","visits_per_month","personal_trainer"
1,2024-04-01,32,Tel Aviv,monthly,250,12,yes
2,2024-09-01,41,Ramat Gan,annual,180,5,no
3,2025-02-01,28,Holon,student,150,18,yes
4,2024-11-01,25,Tel Aviv,monthly,250,10,yes
5,2025-08-01,39,Ramat Gan,annual,180,15,no
6,2024-06-01,22,Holon,monthly,250,8,yes
7,2024-10-01,53,Tel Aviv,annual,180,7,yes
8,2025-03-01,30,Ramat Gan,student,150,14,yes
9,2024-05-01,38,Holon,monthly,250,11,yes
10,2025-01-01,62,Tel Aviv,annual,180,6,yes


In [33]:
batch = pd.read_csv(StringIO(answer))
batch

,member_id,join_date,age,branch,plan,monthly_fee,visits_per_month,personal_trainer
0,1,2024-04-01,32,Tel Aviv,monthly,250,12,yes
1,2,2024-09-01,41,Ramat Gan,annual,180,5,no
2,3,2025-02-01,28,Holon,student,150,18,yes
3,4,2024-11-01,25,Tel Aviv,monthly,250,10,yes
4,5,2025-08-01,39,Ramat Gan,annual,180,15,no
5,6,2024-06-01,22,Holon,monthly,250,8,yes
6,7,2024-10-01,53,Tel Aviv,annual,180,7,yes
7,8,2025-03-01,30,Ramat Gan,student,150,14,yes
8,9,2024-05-01,38,Holon,monthly,250,11,yes
9,10,2025-01-01,62,Tel Aviv,annual,180,6,yes


In [34]:
batches = []

for i in range(10):
    answer = generate_data(request)
    batch = pd.read_csv(StringIO(answer))
    batches.append(batch)
    print(f"Batch {i + 1} done")

members = pd.concat(batches, ignore_index=True)
members

Batch 1 done
Batch 2 done
Batch 3 done
Batch 4 done
Batch 5 done
Batch 6 done
Batch 7 done
Batch 8 done
Batch 9 done
Batch 10 done


,member_id,join_date,age,branch,plan,monthly_fee,visits_per_month,personal_trainer
0,1,2024-06-15,32,Tel Aviv,monthly,250,10,yes
1,2,2024-03-02,41,Ramat Gan,annual,180,5,no
2,3,2024-09-28,55,Holon,student,150,15,yes
3,4,2025-01-10,28,Tel Aviv,monthly,250,12,yes
4,5,2024-11-22,62,Ramat Gan,annual,180,3,no
...,...,...,...,...,...,...,...,...
95,6,2025-02-15,35,Holon,monthly,250,8,no
96,7,2024-07-01,51,Ramat Gan,student,150,5,yes
97,8,2025-04-20,29,Tel Aviv,monthly,250,11,yes
98,9,2024-10-01,44,Holon,annual,180,6,no


In [37]:
members["member_id"] = range(1, len(members) + 1)
members.drop(columns="member_id").duplicated().sum()
(members["visits_per_month"] < 4).sum()

np.int64(3)

In [38]:
members["visits_per_month"].describe()

,visits_per_month
count,100.000000
mean,10.250000
std,3.962896
min,2.000000
25%,7.000000
50%,10.000000
75%,13.250000
max,18.000000


In [39]:
def churn_probability(visits, plan, branch):
    prob = 0.2
    if visits < 7:
        prob += 0.4
    if plan == "annual":
        prob -= 0.1
    if branch == "Holon":
        prob += 0.2
    return round(prob, 2)

print(churn_probability(2, "monthly", "Holon"))
print(churn_probability(10, "annual", "Tel Aviv"))
print(churn_probability(3, "annual", "Ramat Gan"))

0.8
0.1
0.5


In [41]:
import random

random.seed(42)

def decide_churn(prob):
    if random.random() < prob:
        return 1
    return 0

members["churned"] = members.apply(
    lambda row: decide_churn(
        churn_probability(row["visits_per_month"], row["plan"], row["branch"])
    ),
    axis=1,
)

members["churned"].mean()

np.float64(0.36)

In [42]:
members.groupby(members["visits_per_month"] < 7)["churned"].mean()

,churned
visits_per_month,
False,0.328947
True,0.458333


In [43]:
members.groupby("branch")["churned"].mean()

,churned
branch,
Holon,0.500000
Ramat Gan,0.258065
Tel Aviv,0.333333


In [44]:
members.groupby("plan")["churned"].mean()

,churned
plan,
annual,0.218750
monthly,0.380952
student,0.500000


In [45]:
members["plan"].value_counts()

,count
plan,
monthly,42
annual,32
student,26


In [46]:
pd.crosstab(members["plan"], members["branch"])

branch,Holon,Ramat Gan,Tel Aviv
plan,,,
annual,4,22,6
monthly,11,7,24
student,15,2,9


In [47]:
members.groupby("plan")["visits_per_month"].mean()

,visits_per_month
plan,
annual,9.531250
monthly,11.452381
student,9.192308


## Findings

**Main insight:** The Holon branch is the main driver of churn — not student memberships, as the raw numbers first suggest.

**Overall churn:** 36% of members left.

**What drives churn:**
- **Low engagement:** members visiting fewer than 7 times a month churned at 46%, vs. 33% for everyone else.
- **Branch:** Holon churned at 50%, compared to 33% in Tel Aviv and 26% in Ramat Gan.
- **Plan:** annual members churned the least (22%), vs. 38% for monthly.

**The surprise:** student memberships showed the highest churn (50%), even though no rule targeted students.
Digging deeper, 15 of the 26 students belong to the Holon branch. Students don't churn because they are students — they churn because most of them are in the problematic branch.
The same effect makes Ramat Gan look like the best branch: 22 of its 31 members hold annual plans, which naturally churn less.

**Limitations:**
- The data is synthetic: generated by an LLM, with churn assigned by rules I defined.
- With 100 members, some groups are small (e.g., 24 low-visit members), so the exact percentages are noisy.